# TEAR reproduction on Colab
Code: GitHub `TriDang2025/attribute-extraction` (`tear/`). Data and surrogate: Hugging Face. Results: committed back to GitHub (`tear/results/`).

**Before running:** Runtime > Change runtime type > **L4 GPU** (or A100). T4 is too small for the coherence step.

In [ ]:
#@title 1. Tokens, clone/pull, git identity (run every session)
# Needs Colab Secrets (key icon on the left): HF_TOKEN and GH_TOKEN, both with notebook access on.
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

# Safely strip whitespace/newlines from the HF token
hf_token = userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token.strip())

def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
# GitHub token kept in git's in-memory credential cache (no file, never in a URL)
run(["git", "config", "--global", "credential.helper", "cache --timeout=172800"])
run(["git", "credential", "approve"], text=True,
    input=f"""protocol=https
host=github.com
username=token
password={userdata.get('GH_TOKEN').strip() if userdata.get('GH_TOKEN') else ''}

""")
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/tear
!git log -1 --format="code at %h (%an, %ar): %s"

## 2. Setup: packages, data + surrogate from HF, llama-server (CUDA), GGUFs, unit tests
First time on a GPU type ~20-30 min (builds llama-server, then caches it on HF); later ~5 min.

In [ ]:
!bash colab/setup.sh

## 3. Start the model servers

In [ ]:
!bash colab/start_servers.sh

## 4. Run in the background
Start with the 25-text benchmark (5 new texts). Results are committed to GitHub every 30 min and at the end. After a disconnect, run cells 1-3 and the same line again: it resumes from the committed results.

In [ ]:
N = 25  #@param {type:"integer"}
ALL_LEVELS = 1  #@param {type:"integer"}
!bash colab/run.sh {N} {ALL_LEVELS}

## 5. Progress (rerun any time)

In [ ]:
!bash colab/status.sh

## 6. Push results now (optional; the run does this every 30 min)

In [ ]:
!bash colab/sync.sh push-data; bash colab/sync.sh push-results "manual"

## Optional: let Claude Code run it
Run the next cell, then open **Terminal** (bottom-left) and type:
```
export PATH=$HOME/.local/bin:$PATH
cd /content/attribute-extraction/tear
claude
```
Log in when asked (open the link, sign in, paste the code back), then type:
`Read colab/COLAB_AGENT.md and follow it.`
Cell 1 must have run first (it sets up the Hugging Face and GitHub logins the agent uses).

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash